<a href="https://colab.research.google.com/github/Shubham4521/Cost-Aware-LLM-Router/blob/main/notebooks/04_cascade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json, re, subprocess, tempfile, doctest
import pandas as pd
from google.colab import userdata
from openai import OpenAI

OUT = "/content/drive/MyDrive/llm_router"
GITHUB_USER = "Shubham4521"
FREE_ID     = "openai/gpt-oss-120b"
CHEAP_ID    = "openai/gpt-5-mini"
PREMIUM_ID  = "anthropic/claude-sonnet-5"

df = pd.read_csv(
    f"https://raw.githubusercontent.com/{GITHUB_USER}/cost-aware-llm-router/main/data/eval_set.csv",
    keep_default_na=False)
df["needs_search"] = df["needs_search"].astype(str) == "True"

s = pd.read_csv(f"{OUT}/scored_results.csv", keep_default_na=False)
for c in ["score", "cost_usd", "latency_s"]:
    s[c] = s[c].astype(float)
s["passed"] = s["score"] >= 7
lookup = s.set_index(["id", "tier"])
search_cache = json.load(open(f"{OUT}/search_cache.json"))
TIERS = ["free", "cheap", "premium"]

print("Prompts:", len(df), "| Scored rows:", len(s), "| Cached searches:", len(search_cache))

Mounted at /content/drive
Prompts: 165 | Scored rows: 495 | Cached searches: 15


In [3]:
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=userdata.get("GROQ_API_KEY"))
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))

MODELS = {"free": (groq, FREE_ID), "cheap": (openrouter, CHEAP_ID), "premium": (openrouter, PREMIUM_ID)}

def chat(name, messages, max_tokens=2000):
    client, model = MODELS[name]
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens)
    if client is openrouter:
        kwargs["extra_body"] = {"usage": {"include": True}}
    for attempt in range(6):
        try:
            start = time.time()
            r = client.chat.completions.create(**kwargs)
            return {"answer": r.choices[0].message.content or "",
                    "cost_usd": float(getattr(r.usage, "cost", 0) or 0),
                    "latency_s": round(time.time() - start, 2)}
        except Exception as e:
            if "429" in str(e) and attempt < 5:
                wait = 20 * (attempt + 1)
                print(f"Rate limited on {name}, waiting {wait}s")
                time.sleep(wait)
            else:
                raise

print("Models ready")

Models ready


In [4]:
def extract_math(ans):
    m = re.findall(r"ANSWER:\s*(.+)", ans)
    if m:
        return m[-1].strip()
    lines = ans.strip().splitlines()
    return lines[-1] if lines else ""

def norm_math(s):
    s = str(s)
    s = re.sub(r"\\boxed\{(.*)\}", r"\1", s)
    s = re.sub(r"\\text\{(.*?)\}", r"\1", s)
    for t in ["$", "\\left", "\\right", "\\!", " "]:
        s = s.replace(t, "")
    s = s.replace("dfrac", "frac").replace("tfrac", "frac").rstrip(".")
    return s.lower()

def extract_letter(ans):
    a = ans.strip()
    m = re.match(r"^\W*([A-E1-5])\b", a)
    if m: return m.group(1)
    m = re.findall(r"answer\s*(?:is|:)?\s*\(?([A-E1-5])\b", a, re.I)
    if m: return m[-1].upper()
    m = re.findall(r"\b([A-E])\b", a)
    return m[-1] if m else ""

HEADER = "from typing import *\nimport math, re, collections, itertools, heapq, string\n"

def doctest_check(ans, prompt):
    examples = doctest.DocTestParser().get_examples(prompt)
    if not examples:
        return True   # no examples in the prompt -> nothing to check
    blocks = re.findall(r"```(?:python)?\n(.*?)```", ans, re.S)
    code = blocks[0] if blocks else ans
    lines = [HEADER, code]
    for ex in examples:
        lines.append(f"assert ({ex.source.strip()}) == ({ex.want.strip()})")
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write("\n".join(lines))
        path = f.name
    try:
        return subprocess.run(["python", path], capture_output=True, timeout=15).returncode == 0
    except subprocess.TimeoutExpired:
        return False

VERIFY_PROMPT = """Rate this AI answer from 1 to 10 for correctness, completeness and how well it follows the task.
{context}
TASK:
{task}

ANSWER:
{answer}

Reply with only a number from 1 to 10."""

def verify(row, tier):
    ans = str(lookup.loc[(row["id"], tier)]["answer"])
    t = row["answer_type"]
    if t in ("math_answer", "multiple_choice"):
        second = chat(tier, [{"role": "user", "content": row["prompt"]}])
        if t == "math_answer":
            same = norm_math(extract_math(ans)) == norm_math(extract_math(second["answer"]))
        else:
            same = extract_letter(ans) == extract_letter(second["answer"])
        return same, second["cost_usd"], second["latency_s"]
    if t == "code_tests":
        start = time.time()
        ok = doctest_check(ans, row["prompt"])
        return ok, 0.0, round(time.time() - start, 2)
    ctx = ""
    if t == "search_judge":
        ctx = f"SEARCH RESULTS the answer should be based on:\n{search_cache.get(row['prompt'], '')}\n"
    out = chat("cheap", [{"role": "user", "content":
               VERIFY_PROMPT.format(context=ctx, task=row["prompt"], answer=ans)}], max_tokens=500)
    nums = re.findall(r"\b(10|[1-9])\b", out["answer"])
    return (int(nums[0]) if nums else 0) >= 7, out["cost_usd"], out["latency_s"]

print("Checks ready")

Checks ready


In [5]:
VER = f"{OUT}/verify_results.csv"
done = set()
if os.path.exists(VER):
    prev = pd.read_csv(VER, keep_default_na=False)
    done = set(zip(prev["id"], prev["tier"]))

count = 0
for _, row in df.iterrows():
    for tier in ["free", "cheap"]:
        if (row["id"], tier) in done:
            continue
        try:
            verdict, vcost, vlat = verify(row, tier)
        except Exception as e:
            print("FAILED", row["id"], tier, str(e)[:150])
            continue
        rec = {"id": row["id"], "tier": tier, "verdict": verdict, "v_cost": vcost, "v_latency": vlat}
        pd.DataFrame([rec]).to_csv(VER, mode="a", header=not os.path.exists(VER), index=False)
        count += 1
        if count % 10 == 0:
            print(f"{count} checks done")

print("Finished this run. New checks:", count, "| Planned: 330")

10 checks done
20 checks done
30 checks done
40 checks done
50 checks done
60 checks done
70 checks done
80 checks done
90 checks done
100 checks done
110 checks done
120 checks done
130 checks done
140 checks done
150 checks done
160 checks done
170 checks done
180 checks done
190 checks done
200 checks done
210 checks done
220 checks done
230 checks done
240 checks done
250 checks done
260 checks done
270 checks done
280 checks done
290 checks done
300 checks done
310 checks done
320 checks done
330 checks done
Finished this run. New checks: 330 | Planned: 330


In [6]:
SEARCH_RE = re.compile(
    r"\b(today|today's|tonight|latest|most recent|this (week|month|year)|"
    r"currently|current|right now|news|headlines|price of|exchange rate|weather|forecast)\b", re.I)
flag = {r["id"]: bool(SEARCH_RE.search(r["prompt"])) for _, r in df.iterrows()}

v = pd.read_csv(VER, keep_default_na=False)
v["verdict"] = v["verdict"].astype(str) == "True"
for c in ["v_cost", "v_latency"]:
    v[c] = v[c].astype(float)
vlook = v.set_index(["id", "tier"])

def simulate(name, start_tier):
    passed, cost, lat, escalations = 0, 0.0, 0.0, 0
    for _, row in df.iterrows():
        i = row["id"]
        for t in TIERS[TIERS.index(start_tier(row)):]:
            r = lookup.loc[(i, t)]
            cost += r["cost_usd"]
            lat += r["latency_s"]
            final = t
            if t == "premium":
                break
            vv = vlook.loc[(i, t)]
            cost += vv["v_cost"]
            lat += vv["v_latency"]
            if vv["verdict"]:
                break
            escalations += 1
        passed += bool(lookup.loc[(i, final)]["passed"])
    n = len(df)
    return {"router": name, "pass_rate": round(passed / n, 4), "total_cost_usd": round(cost, 4),
            "avg_latency_s": round(lat / n, 2), "escalations": escalations}

prev = pd.read_csv(f"{OUT}/phase3_rule_results.csv")
new = pd.DataFrame([
    simulate("cascade: all start free", lambda row: "free"),
    simulate("cascade: rules start (search->cheap)", lambda row: "cheap" if flag[row["id"]] else "free"),
])
res = pd.concat([prev, new], ignore_index=True)
p = res[res["router"] == "always_premium"].iloc[0]
res["quality_vs_premium_%"] = (res["pass_rate"] / p["pass_rate"] * 100).round(1)
res["cost_vs_premium_%"] = (res["total_cost_usd"] / p["total_cost_usd"] * 100).round(1)
print(res.to_string(index=False))
res.to_csv(f"{OUT}/phase4_cascade_results.csv", index=False)

print("\n=== HOW GOOD IS THE CHECK? (free tier) ===")
m = v[v["tier"] == "free"].merge(s[s["tier"] == "free"][["id", "passed"]], on="id")
print(pd.crosstab(m["verdict"], m["passed"], rownames=["check said OK"], colnames=["actually passed"]))

                              router  pass_rate  total_cost_usd  avg_latency_s  quality_vs_premium_%  cost_vs_premium_%  escalations
                         always_free     0.8970          0.0000           1.33                  96.7                0.0          NaN
                        always_cheap     0.9152          0.1947           8.64                  98.7               42.8          NaN
                      always_premium     0.9273          0.4546           3.29                 100.0              100.0          NaN
                              random     0.8909          0.2168           4.23                  96.1               47.7          NaN
                    oracle (ceiling)     0.9576          0.0266           1.64                 103.3                5.9          NaN
                rules: search->cheap     0.9212          0.0235           2.00                  99.3                5.2          NaN
              rules: search->premium     0.9273          0.1262      